In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("D:\Loan_aproval\Loan_aproval_model\data\loan_data.csv")

<>:1: SyntaxWarning: invalid escape sequence '\L'
<>:1: SyntaxWarning: invalid escape sequence '\L'
C:\Users\user\AppData\Local\Temp\ipykernel_13900\3592621155.py:1: SyntaxWarning: invalid escape sequence '\L'
  df=pd.read_csv("D:\Loan_aproval\Loan_aproval_model\data\loan_data.csv")


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  object 
 2   person_education                45000 non-null  object 
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  object 
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  object 
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_defaults_on_file  

In [4]:
df.columns

Index(['person_age', 'person_gender', 'person_education', 'person_income',
       'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
       'credit_score', 'previous_loan_defaults_on_file', 'loan_status'],
      dtype='object')

In [5]:
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [6]:
df_0=df[df["loan_status"]==0]
df_1=df[df["loan_status"]==1]

In [7]:
df=pd.concat([df_1,df_0[:len(df_1)]])

In [8]:
df=df.sample(frac=1,random_state=42,ignore_index=True)

In [10]:
len(df)

20000

In [11]:
x=df[['person_age', 'person_gender', 'person_education', 'person_income', 'person_home_ownership', 'loan_amnt', 'loan_intent','loan_int_rate', 'cb_person_cred_hist_length','credit_score', 'previous_loan_defaults_on_file']].copy()

In [16]:
x["loan_intent"].unique()

array(['MEDICAL', 'PERSONAL', 'EDUCATION', 'DEBTCONSOLIDATION', 'VENTURE',
       'HOMEIMPROVEMENT'], dtype=object)

In [12]:
y=df["loan_status"].copy()

In [13]:
x_str=x.select_dtypes(exclude="number")
x_int=x.select_dtypes(include="number")
x_str.columns

Index(['person_gender', 'person_education', 'person_home_ownership',
       'loan_intent', 'previous_loan_defaults_on_file'],
      dtype='object')

In [14]:
x_int.columns

Index(['person_age', 'person_income', 'loan_amnt', 'loan_int_rate',
       'cb_person_cred_hist_length', 'credit_score'],
      dtype='object')

In [15]:
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [16]:
preprocess=ColumnTransformer(
    transformers=[
        ("onehotenc",OneHotEncoder(),['person_gender', 'person_home_ownership','loan_intent', 'previous_loan_defaults_on_file']),
        ("education",OrdinalEncoder(categories=[['High School', 'Associate', 'Bachelor', 'Master', 'Doctorate']]),["person_education"]),
        ("standard",StandardScaler(),x_int.columns)
    ],
    remainder="passthrough"
)

In [17]:
preprocess.fit(x)
new_x=preprocess.transform(x)

In [18]:
len(preprocess.get_feature_names_out())

21

In [19]:
new_x=pd.DataFrame(new_x,columns=preprocess.get_feature_names_out())
new_data=pd.concat([new_x,y],axis=1)

In [20]:
y.value_counts()

loan_status
0    10000
1    10000
Name: count, dtype: int64

In [21]:
new_data.head()

,onehotenc__person_gender_female,onehotenc__person_gender_male,onehotenc__person_home_ownership_MORTGAGE,onehotenc__person_home_ownership_OTHER,onehotenc__person_home_ownership_OWN,onehotenc__person_home_ownership_RENT,onehotenc__loan_intent_DEBTCONSOLIDATION,onehotenc__loan_intent_EDUCATION,onehotenc__loan_intent_HOMEIMPROVEMENT,onehotenc__loan_intent_MEDICAL,...,onehotenc__previous_loan_defaults_on_file_No,onehotenc__previous_loan_defaults_on_file_Yes,education__person_education,standard__person_age,standard__person_income,standard__loan_amnt,standard__loan_int_rate,standard__cb_person_cred_hist_length,standard__credit_score,loan_status
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,1.0,3.0,0.084187,-0.711438,-1.338215,-1.221140,-0.120779,-0.045601,0
1,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,-0.716453,-0.418354,-0.397238,-0.857148,-0.439207,-0.065587,1
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.484508,-1.238104,-1.169310,-1.211726,0.516075,1.373430,1
3,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,3.0,-0.716453,-0.704546,-0.318823,-0.486880,-0.120779,-0.365382,1
4,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,-0.516293,0.365565,-0.710897,0.736887,-0.757634,-0.385369,0
